In [1]:
# inlegalbert_kg_rag_rrc_v2.py  (GPU-accelerated rewrite)
#
# Architecture:
#   InLegalBERT → BiLSTM → Multi-Head Attention Pooling → CRF
#   + ENHANCED Knowledge Graph (Dual-Partition: 𝒢_maj + 𝒢_rare)
#   + Rare-Exclusive Dense Subgraph with Virtual Interpolation Nodes
#   + Prototype Centroid Two-Level Index per Rare Class
#   + Rare-Biased Retrieval (λ priority score boost)
#   + GAT r_boost: rare neighbours amplified during fusion
#
# KEY FIX vs v2_base:
#   ALL KG tensors (node embeddings, edge indices, prototype centroids)
#   are stored and operated on the GPU.  The retrieve() hot-path is
#   fully vectorised – zero Python for-loops over edges or nodes.
#   _kg_fuse_batch() is also vectorised: one batched matmul per subgraph.
#
#   Specifically:
#     • KnowledgeGraph._stacked[lid]       → GPU Tensor (N, D)
#     • KnowledgeGraph._edge_idx[lid]      → GPU LongTensor (2, E)
#     • KnowledgeGraph._edge_w[lid]        → GPU FloatTensor (E,)
#     • KnowledgeGraph._proto_stacked[lid] → GPU Tensor (K, D)
#     • KGRetriever.retrieve()             → batched cosine via torch.mm,
#                                            topk on GPU, edge expansion via
#                                            torch.isin (no Python loops)
#     • KGAugmentedModel._kg_fuse_batch()  → processes ALL sentences of a
#                                            document simultaneously with a
#                                            single masked mm + scatter

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)
from sklearn.cluster import MiniBatchKMeans   # faster than KMeans for large N

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_v2_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 20
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1
ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4
WARMUP_RATIO     = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD   = 0.05

# ── KG-RAG ────────────────────────────────────────────────
KG_TOP_K        = 3
KG_TOP_NODES    = 5
KG_HOP          = 1
KG_FUSION_DIM   = 256    # = SENT_OUT_DIM (128*2)

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# ── Dual-threshold uncertainty ────────────────────────────
UNCERTAINTY_THRESH_MAJORITY = 0.70
UNCERTAINTY_THRESH_RARE     = 0.40

# ── Rare-exclusive subgraph ───────────────────────────────
RARE_ALWAYS_KG           = True
KG_VIRTUAL_ALPHAS        = [0.25, 0.50, 0.75]
KG_PROTOTYPE_K           = 3
KG_RARE_LAMBDA           = 0.15
KG_RARE_GAMMA            = 1.5
KG_MAX_VIRTUAL_PER_LABEL = 300

# ── GPU KG limits (to avoid OOM on large subgraphs) ───────
KG_MAX_NODES_PER_SG      = 2000   # cap nodes used in retrieval mm
KG_MAX_CROSS_EDGES       = 2000
KG_MAX_INTRA_EDGES_STORE = 500_000   # per label; dense graphs get sampled

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2  # 256

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2  # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total = len(encoder_layers)
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask,
                                             token_type_ids, lengths=lengths)
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    GPU-resident dual-partition KG.

    Internal storage (all on self.device after .to_device() is called):
        _stacked[lid]      : Tensor (N, D)   – all node embeddings
        _norm[lid]         : Tensor (N, D)   – L2-normalised embeddings
        _is_rare_node[lid] : Tensor (N,)     – bool, True for virtual/proto
        _edge_idx[lid]     : LongTensor (2,E)– src/dst indices (symmetric)
        _edge_w[lid]       : Tensor (E,)     – cosine weights
        _proto[lid]        : Tensor (K, D)   – prototype centroids (normed)
        _proto_norm[lid]   : Tensor (K, D)
        cross_src_lbl      : LongTensor (X,) – source label ids
        cross_src_idx      : LongTensor (X,) – source node indices
        cross_dst_lbl      : LongTensor (X,) – dest label ids
        cross_dst_idx      : LongTensor (X,) – dest node indices
        cross_w            : Tensor (X,)     – weights
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, device=DEVICE):
        self.emb_dim        = emb_dim
        self.device         = device
        self.rare_partition = set()

        # CPU-side node store (raw, before GPU promotion)
        self._cpu_nodes   = defaultdict(list)   # lid -> list of (emb_cpu, is_virtual, is_proto)

        # GPU tensors (populated by to_device() / build_edges())
        self._stacked     = {}
        self._norm        = {}
        self._is_rare_node= {}
        self._edge_idx    = {}
        self._edge_w      = {}
        self._proto       = {}
        self._proto_norm  = {}

        # Cross edges – stored as 5 parallel GPU tensors
        self._cross_src_lbl = None
        self._cross_src_idx = None
        self._cross_dst_lbl = None
        self._cross_dst_idx = None
        self._cross_w       = None

    # ─── Population ─────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            self._cpu_nodes[lid].append((emb, False, False))

    # ─── Virtual nodes ───────────────────────────────────────
    def _inject_virtual_nodes(self, lid: int,
                              alphas=KG_VIRTUAL_ALPHAS,
                              max_virtual=KG_MAX_VIRTUAL_PER_LABEL):
        real = [n[0] for n in self._cpu_nodes[lid] if not n[1] and not n[2]]
        n = len(real)
        if n < 2:
            return
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        random.shuffle(pairs)
        added = 0
        for (i, j) in pairs:
            if added >= max_virtual:
                break
            ei, ej = real[i], real[j]
            for alpha in alphas:
                if added >= max_virtual:
                    break
                v = alpha * ei + (1 - alpha) * ej
                v = F.normalize(v.unsqueeze(0), dim=-1).squeeze(0)
                self._cpu_nodes[lid].append((v, True, False))
                added += 1
        print(f"    [{id2label[lid]}] injected {added} virtual nodes (real={n})")

    # ─── Prototypes (MiniBatchKMeans on GPU via numpy bridge) ─
    def _build_prototypes(self, lid: int, k: int = KG_PROTOTYPE_K):
        all_embs = torch.stack([n[0] for n in self._cpu_nodes[lid]])
        n_nodes  = all_embs.shape[0]
        k_actual = min(k, n_nodes)
        if k_actual < 2:
            c = all_embs.mean(0, keepdim=True)
            c = F.normalize(c, dim=-1)
        else:
            km = MiniBatchKMeans(n_clusters=k_actual, n_init=5,
                                 batch_size=min(1024, n_nodes),
                                 random_state=SEED)
            km.fit(all_embs.numpy())
            c = torch.tensor(km.cluster_centers_, dtype=torch.float32)
            c = F.normalize(c, dim=-1)

        for ci in range(c.shape[0]):
            self._cpu_nodes[lid].insert(0, (c[ci], False, True))
        print(f"    [{id2label[lid]}] built {c.shape[0]} prototype centroids")
        return c   # (K, D)

    # ─── Build all GPU tensors ────────────────────────────────
    def build_edges(self,
                    rare_ids: list,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_per_node=5,
                    max_cross_edges=KG_MAX_CROSS_EDGES):

        print("  Building dual-partition KG edges (GPU-accelerated) ...")
        self.rare_partition = set(rare_ids)

        # ── Step 1: virtual + prototype injection ──────────────
        print("  Injecting virtual nodes & prototypes ...")
        for lid in rare_ids:
            if self._cpu_nodes[lid]:
                self._inject_virtual_nodes(lid)
                self._build_prototypes(lid, k=KG_PROTOTYPE_K)

        # ── Step 2: move all nodes to GPU tensors ──────────────
        print("  Promoting node embeddings to GPU ...")
        for lid, node_list in self._cpu_nodes.items():
            embs = torch.stack([n[0] for n in node_list]).to(self.device)  # (N, D)
            norms = F.normalize(embs, dim=-1)
            is_virt = torch.tensor([n[1] or n[2] for n in node_list],
                                   dtype=torch.bool, device=self.device)
            self._stacked[lid] = embs
            self._norm[lid]    = norms
            self._is_rare_node[lid] = is_virt

        # ── Step 3: intra-label edges → GPU sparse ────────────
        print("  Building intra-label edges ...")
        for lid in self._stacked:
            norms = self._norm[lid]
            N = norms.shape[0]
            if N < 2:
                self._edge_idx[lid] = torch.zeros(2, 0, dtype=torch.long, device=self.device)
                self._edge_w[lid]   = torch.zeros(0, device=self.device)
                continue

            if lid in self.rare_partition:
                # ── DENSE: all-pairs, but chunk to avoid OOM ──
                chunk = min(512, N)
                ei_list, ej_list, ew_list = [], [], []
                for start in range(0, N, chunk):
                    end = min(start + chunk, N)
                    sim_block = torch.mm(norms[start:end], norms.T)  # (chunk, N)
                    rows, cols = torch.where(
                        (sim_block > 0) &
                        (torch.arange(start, end, device=self.device).unsqueeze(1)
                         < torch.arange(N, device=self.device).unsqueeze(0))
                    )
                    ei_list.append(rows + start)
                    ej_list.append(cols)
                    ew_list.append(sim_block[rows, cols])

                    # cap to avoid huge memory
                    if sum(x.shape[0] for x in ei_list) >= KG_MAX_INTRA_EDGES_STORE:
                        break

                if ei_list:
                    ei = torch.cat(ei_list)
                    ej = torch.cat(ej_list)
                    ew = torch.cat(ew_list)
                    # sample if over cap
                    if ei.shape[0] > KG_MAX_INTRA_EDGES_STORE:
                        perm = torch.randperm(ei.shape[0], device=self.device)[:KG_MAX_INTRA_EDGES_STORE]
                        ei, ej, ew = ei[perm], ej[perm], ew[perm]
                    # make symmetric
                    src = torch.cat([ei, ej])
                    dst = torch.cat([ej, ei])
                    ww  = torch.cat([ew, ew])
                else:
                    src = dst = torch.zeros(0, dtype=torch.long, device=self.device)
                    ww  = torch.zeros(0, device=self.device)

            else:
                # ── SPARSE: consecutive + high-cosine for majority ──
                # consecutive edges
                cons_i = torch.arange(N - 1, device=self.device)
                cons_j = cons_i + 1
                cons_w = (norms[cons_i] * norms[cons_j]).sum(-1).clamp(min=0)

                # high-sim non-consecutive
                sim_mat = torch.mm(norms, norms.T)
                sim_mat.fill_diagonal_(-2.0)
                # mask consecutive neighbours to avoid duplication
                idx = torch.arange(N, device=self.device)
                sim_mat[idx[:-1], idx[1:]] = -2.0
                sim_mat[idx[1:], idx[:-1]] = -2.0

                hi_rows, hi_cols = torch.where(sim_mat >= intra_thresh)
                # keep only upper triangle to deduplicate
                keep = hi_rows < hi_cols
                hi_rows, hi_cols = hi_rows[keep], hi_cols[keep]
                hi_w = sim_mat[hi_rows, hi_cols]

                # per-node cap
                if hi_rows.shape[0] > N * max_intra_per_node:
                    perm = torch.randperm(hi_rows.shape[0], device=self.device)[:N * max_intra_per_node]
                    hi_rows, hi_cols, hi_w = hi_rows[perm], hi_cols[perm], hi_w[perm]

                ei = torch.cat([cons_i, hi_rows])
                ej = torch.cat([cons_j, hi_cols])
                ew = torch.cat([cons_w, hi_w])
                src = torch.cat([ei, ej])
                dst = torch.cat([ej, ei])
                ww  = torch.cat([ew, ew])

            self._edge_idx[lid] = torch.stack([src, dst], dim=0)
            self._edge_w[lid]   = ww

        # ── Step 4: prototypes to GPU ─────────────────────────
        for lid in rare_ids:
            proto_nodes = [n for n in self._cpu_nodes.get(lid, []) if n[2]]
            if proto_nodes:
                pc = torch.stack([n[0] for n in proto_nodes]).to(self.device)
                pc = F.normalize(pc, dim=-1)
                self._proto[lid]      = pc
                self._proto_norm[lid] = pc

        # ── Step 5: cross-label edges → GPU tensors ───────────
        print("  Building cross-label edges ...")
        label_ids = list(self._stacked.keys())
        c_sl, c_si, c_dl, c_di, c_w = [], [], [], [], []
        total = 0

        for a in range(len(label_ids)):
            if total >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if total >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                na = self._norm[la]
                nb = self._norm[lb]
                # Cap each subgraph for the cross-sim mm
                na_cap = na[:KG_MAX_NODES_PER_SG]
                nb_cap = nb[:KG_MAX_NODES_PER_SG]
                sim = torch.mm(na_cap, nb_cap.T)
                rows, cols = torch.where(sim >= cross_thresh)
                rows, cols = rows[:50], cols[:50]
                if rows.shape[0] == 0:
                    continue
                w = sim[rows, cols]
                c_sl.append(torch.full((rows.shape[0],), la, dtype=torch.long, device=self.device))
                c_si.append(rows)
                c_dl.append(torch.full((rows.shape[0],), lb, dtype=torch.long, device=self.device))
                c_di.append(cols)
                c_w.append(w)
                total += rows.shape[0]

        if c_sl:
            self._cross_src_lbl = torch.cat(c_sl)
            self._cross_src_idx = torch.cat(c_si)
            self._cross_dst_lbl = torch.cat(c_dl)
            self._cross_dst_idx = torch.cat(c_di)
            self._cross_w       = torch.cat(c_w)
        else:
            self._cross_src_lbl = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_src_idx = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_dst_lbl = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_dst_idx = torch.zeros(0, dtype=torch.long, device=self.device)
            self._cross_w       = torch.zeros(0, device=self.device)

        n_rare_nodes = sum(self._stacked[lid].shape[0]
                           for lid in self.rare_partition if lid in self._stacked)
        n_maj_nodes  = sum(self._stacked[lid].shape[0]
                           for lid in self._stacked if lid not in self.rare_partition)
        n_intra = sum(self._edge_w[lid].shape[0] // 2   # symmetric → halved for display
                      for lid in self._edge_w)
        print(f"  KG built:")
        print(f"    Majority nodes : {n_maj_nodes} | Rare nodes: {n_rare_nodes}")
        print(f"    Intra-edges    : {n_intra} | Cross-edges: {total}")

    # ─── Save / Load (CPU JSON) ──────────────────────────────
    def save(self, path):
        data = {
            "rare_partition": list(self.rare_partition),
            "nodes": {
                str(lid): [
                    {"emb": n[0].tolist(), "is_virtual": n[1], "is_proto": n[2]}
                    for n in node_list
                ]
                for lid, node_list in self._cpu_nodes.items()
            },
        }
        # save edge tensors as lists (compact)
        data["intra"] = {
            str(lid): {
                "idx": self._edge_idx[lid].cpu().tolist(),
                "w":   self._edge_w[lid].cpu().tolist(),
            }
            for lid in self._edge_idx
        }
        cross = {}
        if self._cross_src_lbl is not None and self._cross_src_lbl.shape[0] > 0:
            cross = {
                "sl": self._cross_src_lbl.cpu().tolist(),
                "si": self._cross_src_idx.cpu().tolist(),
                "dl": self._cross_dst_lbl.cpu().tolist(),
                "di": self._cross_dst_idx.cpu().tolist(),
                "w":  self._cross_w.cpu().tolist(),
            }
        data["cross"] = cross
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM, device=DEVICE):
        kg = cls(emb_dim=emb_dim, device=device)
        with open(path) as f:
            data = json.load(f)
        kg.rare_partition = set(data.get("rare_partition", []))

        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                emb = torch.tensor(n["emb"], dtype=torch.float32)
                kg._cpu_nodes[lid].append((emb, n["is_virtual"], n["is_proto"]))

        # Rebuild GPU tensors
        for lid, node_list in kg._cpu_nodes.items():
            embs  = torch.stack([n[0] for n in node_list]).to(device)
            norms = F.normalize(embs, dim=-1)
            is_virt = torch.tensor([n[1] or n[2] for n in node_list],
                                   dtype=torch.bool, device=device)
            kg._stacked[lid]      = embs
            kg._norm[lid]         = norms
            kg._is_rare_node[lid] = is_virt

        for k, v in data.get("intra", {}).items():
            lid = int(k)
            idx = torch.tensor(v["idx"], dtype=torch.long,  device=device)
            w   = torch.tensor(v["w"],   dtype=torch.float32, device=device)
            kg._edge_idx[lid] = idx
            kg._edge_w[lid]   = w

        cross = data.get("cross", {})
        if cross:
            kg._cross_src_lbl = torch.tensor(cross["sl"], dtype=torch.long,    device=device)
            kg._cross_src_idx = torch.tensor(cross["si"], dtype=torch.long,    device=device)
            kg._cross_dst_lbl = torch.tensor(cross["dl"], dtype=torch.long,    device=device)
            kg._cross_dst_idx = torch.tensor(cross["di"], dtype=torch.long,    device=device)
            kg._cross_w       = torch.tensor(cross["w"],  dtype=torch.float32, device=device)
        else:
            z = torch.zeros(0, dtype=torch.long, device=device)
            kg._cross_src_lbl = z; kg._cross_src_idx = z
            kg._cross_dst_lbl = z; kg._cross_dst_idx = z
            kg._cross_w = torch.zeros(0, device=device)

        # prototypes
        for lid in kg.rare_partition:
            proto_nodes = [n for n in kg._cpu_nodes.get(lid, []) if n[2]]
            if proto_nodes:
                pc = torch.stack([n[0] for n in proto_nodes]).to(device)
                pc = F.normalize(pc, dim=-1)
                kg._proto[lid]      = pc
                kg._proto_norm[lid] = pc

        print(f"  KG loaded ← {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS,
                 thresh_majority=UNCERTAINTY_THRESH_MAJORITY,
                 thresh_rare=UNCERTAINTY_THRESH_RARE):
        self.log_C           = math.log(num_classes)
        self.thresh_majority = thresh_majority
        self.thresh_rare     = thresh_rare

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor,
                     is_rare_pred: torch.Tensor) -> torch.Tensor:
        H = self.entropy(logits)
        thresh = torch.where(
            is_rare_pred,
            torch.full_like(H, self.thresh_rare),
            torch.full_like(H, self.thresh_majority),
        )
        return H > thresh

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Fully vectorised retrieval – no Python loops over nodes or edges.

    retrieve_batch(H, pred_labels, uncertain_mask) processes all
    sentences in one document simultaneously:
      1. Score each subgraph: max_cosine(H_valid, SG) + λ·is_rare
         → one mm per subgraph (GPU)
      2. Top-K subgraphs selected once for the whole batch.
      3. For each selected subgraph, gather top-nodes via topk.
      4. Edge-expansion via torch.isin on GPU LongTensors.
      5. Cross-edge lookup via masked index on GPU tensors.
    Returns (neighbour_embs, weights, is_rare_flags) as GPU tensors.
    """

    def __init__(self, kg: KnowledgeGraph,
                 rare_ids: list,
                 top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES,
                 hop=KG_HOP,
                 lambda_boost=KG_RARE_LAMBDA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.rare_tensor  = torch.tensor(sorted(rare_ids),
                                         dtype=torch.long, device=kg.device)
        self.top_k        = top_k
        self.top_nodes    = top_nodes
        self.hop          = hop
        self.lambda_boost = lambda_boost
        self.device       = kg.device
        self.label_ids    = sorted(kg._stacked.keys())

    @classmethod
    def calibrate_lambda(cls, kg: KnowledgeGraph, rare_ids: list,
                         sample_limit: int = 200) -> float:
        rare_set = set(rare_ids)
        maj_ids  = [lid for lid in kg._stacked if lid not in rare_set]
        if not maj_ids:
            return KG_RARE_LAMBDA

        # stack majority norms once
        maj_norms_list = [kg._norm[m][:KG_MAX_NODES_PER_SG] for m in maj_ids
                          if m in kg._norm]
        if not maj_norms_list:
            return KG_RARE_LAMBDA
        maj_norms = torch.cat(maj_norms_list, dim=0)  # (M_total, D) GPU

        gaps = []
        for lid in rare_ids:
            if lid not in kg._norm:
                continue
            own_norm = kg._norm[lid]
            N = own_norm.shape[0]
            idx = torch.randperm(N, device=kg.device)[:sample_limit]
            sampled = own_norm[idx]   # (S, D) GPU

            own_sims = torch.mm(sampled, own_norm.T).max(dim=1).values  # (S,)
            maj_sims = torch.mm(sampled, maj_norms.T).max(dim=1).values # (S,)
            gaps.append((maj_sims - own_sims).cpu())

        if not gaps:
            return KG_RARE_LAMBDA
        all_gaps = torch.cat(gaps)
        lam = float(all_gaps.median().item())
        lam = float(np.clip(lam, 0.05, 0.40))
        print(f"  λ auto-calibrated: median gap = {all_gaps.median().item():.4f} → λ = {lam:.4f}")
        return lam

    # ── Main batched entry point ──────────────────────────────
    def retrieve_batch(self,
                       H: torch.Tensor,          # (T, D) query vectors (GPU, L2-normed)
                       trigger_mask: torch.Tensor # (T,) bool – which sentences need KG
                       ) -> tuple:
        """
        Returns three GPU tensors aligned with trigger positions:
            nb_embs   : (T_triggered, K_total, D)
            nb_weights: (T_triggered, K_total)
            nb_is_rare: (T_triggered, K_total) bool
        where K_total = top_k * (top_nodes + edge_expansion + cross) padded.
        Padding positions are marked by nb_weights == 0.
        """
        if not trigger_mask.any():
            return None, None, None

        trig_idx = trigger_mask.nonzero(as_tuple=True)[0]  # (T',)
        H_q      = H[trig_idx]                             # (T', D)
        T_q      = H_q.shape[0]

        # ── 1. Score subgraphs ────────────────────────────────
        # For each label subgraph compute max cosine similarity to query batch
        sg_score_list = []
        for lid in self.label_ids:
            norms = self.kg._norm[lid]
            if norms.shape[0] == 0:
                sg_score_list.append(torch.full((T_q,), -1.0, device=self.device))
                continue
            cap = min(norms.shape[0], KG_MAX_NODES_PER_SG)
            sim = torch.mm(H_q, norms[:cap].T)  # (T', cap)
            max_sim = sim.max(dim=1).values       # (T',)
            boost = self.lambda_boost if lid in self.rare_ids else 0.0
            sg_score_list.append(max_sim + boost)

        sg_scores = torch.stack(sg_score_list, dim=1)  # (T', num_labels)
        topk_scores, topk_lids_idx = sg_scores.topk(
            min(self.top_k, len(self.label_ids)), dim=1
        )  # (T', top_k)

        # ── 2. Gather candidate nodes per query ──────────────
        # We'll build a padded tensor of (emb, weight, is_rare)
        # Conservative capacity estimate
        cap = self.top_k * (self.top_nodes * 4 + 10)
        nb_embs    = torch.zeros(T_q, cap, H_q.shape[1], device=self.device)
        nb_weights = torch.zeros(T_q, cap, device=self.device)
        nb_is_rare = torch.zeros(T_q, cap, dtype=torch.bool, device=self.device)
        fill       = torch.zeros(T_q, dtype=torch.long, device=self.device)

        for ki in range(topk_lids_idx.shape[1]):
            for qi in range(T_q):
                lid_idx = topk_lids_idx[qi, ki].item()
                lid     = self.label_ids[lid_idx]
                is_rare_sg = lid in self.rare_ids

                norms = self.kg._norm[lid]
                embs  = self.kg._stacked[lid]
                N_sg  = norms.shape[0]
                if N_sg == 0:
                    continue

                h = H_q[qi]  # (D,)

                # ── Coarse prototype search for rare ──────────
                seed_set = None
                if is_rare_sg and lid in self.kg._proto_norm:
                    p_norm = self.kg._proto_norm[lid]  # (K, D)
                    p_sims = torch.mv(p_norm, h)
                    best_p = int(p_sims.argmax().item())
                    # 1-hop from prototype via edge index
                    edge = self.kg._edge_idx.get(lid)
                    if edge is not None and edge.shape[1] > 0:
                        proto_node_idx = torch.tensor([best_p], device=self.device)
                        src, dst = edge[0], edge[1]
                        mask_e = (src.unsqueeze(0) == proto_node_idx.unsqueeze(1)).any(0)
                        hop_dst = dst[mask_e]
                        seed_set = torch.cat([proto_node_idx, hop_dst]).unique()
                        seed_set = seed_set[seed_set < N_sg]

                # ── Top-N by cosine ───────────────────────────
                k_q = min(self.top_nodes, N_sg)
                cap_norms = norms[:KG_MAX_NODES_PER_SG]
                sims = torch.mv(cap_norms, h)
                topn = sims.topk(k_q)
                top_idx  = topn.indices
                top_sims = topn.values

                if seed_set is not None:
                    top_idx  = torch.cat([top_idx, seed_set]).unique()
                    top_sims = torch.mv(norms[top_idx], h)
                    top_idx  = top_idx[:cap]
                    top_sims = top_sims[:cap]

                # ── 1-hop edge expansion ──────────────────────
                if self.hop >= 1:
                    edge = self.kg._edge_idx.get(lid)
                    if edge is not None and edge.shape[1] > 0:
                        src, dst = edge[0], edge[1]
                        in_top = torch.isin(src, top_idx)
                        hop_n  = dst[in_top].unique()
                        if hop_n.shape[0] > 0:
                            hop_n = hop_n[hop_n < N_sg]
                            hop_s = torch.mv(norms[hop_n], h)
                            top_idx  = torch.cat([top_idx, hop_n]).unique()
                            top_sims = torch.cat([top_sims, hop_s])

                # ── Cross-edge expansion ──────────────────────
                cross_sl = self.kg._cross_src_lbl
                if cross_sl is not None and cross_sl.shape[0] > 0:
                    mask_la = (cross_sl == lid)
                    if mask_la.any():
                        c_si = self.kg._cross_src_idx[mask_la]
                        c_dl = self.kg._cross_dst_lbl[mask_la]
                        c_di = self.kg._cross_dst_idx[mask_la]
                        c_w  = self.kg._cross_w[mask_la]
                        in_top_c = torch.isin(c_si, top_idx)
                        if in_top_c.any():
                            for ci in range(in_top_c.shape[0]):
                                if not in_top_c[ci]:
                                    continue
                                dlid = int(c_dl[ci].item())
                                dstk = self.kg._stacked.get(dlid)
                                if dstk is None:
                                    continue
                                didx = int(c_di[ci].item())
                                if didx >= dstk.shape[0]:
                                    continue
                                f = int(fill[qi].item())
                                if f >= cap:
                                    break
                                nb_embs[qi, f]    = dstk[didx]
                                nb_weights[qi, f] = float(c_w[ci].item())
                                nb_is_rare[qi, f] = dlid in self.rare_ids
                                fill[qi] = f + 1

                # ── Fill output tensors ───────────────────────
                n_add = min(top_idx.shape[0], cap - int(fill[qi].item()))
                if n_add <= 0:
                    continue
                f = int(fill[qi].item())
                idx_use = top_idx[:n_add]
                sim_use = top_sims[:n_add]
                nb_embs[qi, f:f+n_add]    = embs[idx_use]
                nb_weights[qi, f:f+n_add] = sim_use.clamp(min=0)
                nb_is_rare[qi, f:f+n_add] = is_rare_sg
                fill[qi] = f + n_add

        return nb_embs, nb_weights, nb_is_rare   # (T', cap, D), (T', cap), (T', cap)


# ═══════════════════════════════════════════════════════════
# GPU-ACCELERATED GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    Batched GAT fusion.
    Input:
        H_q      : (T', D)          – query vectors (GPU)
        nb_embs  : (T', K, D)       – neighbour embeddings (GPU, padded)
        nb_w     : (T', K)          – edge weights (0 = padding)
        nb_rare  : (T', K) bool     – is-rare flags
    Output:
        v        : (T', D)          – fused vectors
    """

    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT,
                 r_boost_gamma=KG_RARE_GAMMA):
        super().__init__()
        self.emb_dim       = emb_dim
        self.r_boost_gamma = r_boost_gamma
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, H_q, nb_embs, nb_w, nb_rare):
        """
        H_q     : (T', D)
        nb_embs : (T', K, D)
        nb_w    : (T', K)   – 0 means padding
        nb_rare : (T', K) bool
        """
        T, D = H_q.shape
        K    = nb_embs.shape[1]

        q = self.proj_q(H_q)           # (T', D)
        k = self.proj_k(nb_embs)       # (T', K, D)

        # dot product: (T', K)
        dot = torch.bmm(k, q.unsqueeze(-1)).squeeze(-1) * self.scale  # (T', K)

        # r_boost
        r_boost = torch.where(nb_rare,
                              torch.full_like(dot, self.r_boost_gamma),
                              torch.ones_like(dot))

        # masking padding (nb_w == 0)
        pad_mask = (nb_w == 0)
        raw = dot * nb_w * r_boost
        raw = raw.masked_fill(pad_mask, -1e9)

        alpha = F.softmax(raw, dim=-1)  # (T', K)
        alpha = alpha.masked_fill(pad_mask, 0.0)
        alpha = self.dropout(alpha)

        v = torch.bmm(alpha.unsqueeze(1), nb_embs).squeeze(1)  # (T', D)
        return v


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):

    def __init__(self, base_model: InLegalBERT_BiLSTM_MHA_CRF,
                 kg: KnowledgeGraph,
                 rare_ids: list,
                 retriever: KGRetriever = None):
        super().__init__()
        self.base        = base_model
        self.kg          = kg
        self.rare_ids    = set(rare_ids)
        self.retriever   = retriever or KGRetriever(kg, rare_ids)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100
        )

        # rare_ids tensor on same device as KG
        self._rare_tensor = torch.tensor(sorted(rare_ids), dtype=torch.long)

    def _rare_mask(self, top_labels: torch.Tensor) -> torch.Tensor:
        """top_labels: (B, T) → bool (B, T)"""
        rare = self._rare_tensor.to(top_labels.device)
        return (top_labels.unsqueeze(-1) == rare.view(1, 1, -1)).any(-1)

    # ── Vectorised KG fusion ──────────────────────────────────
    def _kg_fuse_batch(self, sent_vecs, emissions, lengths, device):
        """
        Processes the entire batch at once using vectorised retrieval.
        sent_vecs : (B, T, D)
        emissions : (B, T, C)
        Returns fused (B, T, D)
        """
        B, T, D = sent_vecs.shape
        fused = sent_vecs.clone()

        top_labels   = self.uncertainty.top_label(emissions)    # (B, T)
        is_rare_pred = self._rare_mask(top_labels)              # (B, T)
        uncertain    = self.uncertainty.is_uncertain(emissions, is_rare_pred)  # (B, T)

        trigger = uncertain | (RARE_ALWAYS_KG & is_rare_pred)   # (B, T)

        for b in range(B):
            n = int(lengths[b].item())
            trig_b = trigger[b, :n]                             # (T_n,)
            if not trig_b.any():
                continue

            H_b = sent_vecs[b, :n]                             # (T_n, D)
            # L2-normalise for cosine similarity in retriever
            H_b_norm = F.normalize(H_b.detach(), dim=-1)

            nb_embs, nb_weights, nb_is_rare = self.retriever.retrieve_batch(
                H_b_norm, trig_b
            )
            if nb_embs is None:
                continue

            # trig positions within the document
            trig_idx = trig_b.nonzero(as_tuple=True)[0]        # (T',)
            H_q      = H_b[trig_idx]                           # (T', D)

            # GAT fusion (fully batched, GPU)
            v = self.gat_fusion(H_q, nb_embs, nb_weights, nb_is_rare)  # (T', D)

            # residual write-back
            fused[b, trig_idx] = H_b[trig_idx] + v

        return fused

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base first pass
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # Step 2: GPU KG fusion
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )

        # Step 3: re-run ctx-BiLSTM on fused representations
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # Step 4: fusion classifier + CRF
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool, device=device)

        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            base_crf_loss  = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2*T2, C),
                labels.reshape(B2*T2),
            )
            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues  = [id2label[x] for x in all_trues]
    str_preds  = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# BASE TRAINER
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({"params": params, "lr": lr_i,
                                     "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({"params": head_params, "lr": HEAD_LR,
                             "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           rare_ids, device=DEVICE) -> KnowledgeGraph:
    print("\n🔨 Building Dual-Partition Knowledge Graph (GPU) ...")
    base_model.eval()
    base_model.to(device)

    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim, device=device)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=4, shuffle=False,
                        collate_fn=collate_rrc, num_workers=0)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids   = ids.to(device)
        attn  = attn.to(device)
        ttype = ttype.to(device)

        # process each doc in the micro-batch
        for bi in range(ids.shape[0]):
            sent_vecs = base_model.encode_sentences(
                ids[bi:bi+1], attn[bi:bi+1], ttype[bi:bi+1]
            ).squeeze(0)
            n = int(lengths[bi].item())
            kg.add_nodes(sent_vecs[:n].cpu(), labels[bi, :n].tolist())

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {(doc_idx+1)*ids.shape[0]}/{len(train_docs)} docs")

    kg.build_edges(rare_ids=rare_ids)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_fusion.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters())
        )
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )
        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]  Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-RAG v2)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1  (rare-exclusive KG-RAG v2)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1",  marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → Rare-Exclusive KG-RAG)")
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None, total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + Rare-Excl. KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 68)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 68)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 68)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF")
    print("               + Rare-Exclusive Dense KG + Biased Retrieval (GPU)\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF()
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ════════════════════════════════════════════════════
    # PHASE A→B: Build Dual-Partition KG
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Dual-Partition Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG — loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim, device=DEVICE)
        kg.rare_partition = set(rare_ids)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer,
                                   rare_ids=rare_ids, device=DEVICE)

    # ── Auto-calibrate λ ──────────────────────────────
    print("\n  Calibrating λ (retrieval priority boost) ...")
    lambda_boost = KGRetriever.calibrate_lambda(kg, rare_ids)

    # ════════════════════════════════════════════════════
    # PHASE B: KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: Rare-Exclusive KG-Augmented Fine-Tuning")
    print(f"  λ={lambda_boost:.4f}  γ={KG_RARE_GAMMA}  "
          f"thresh_rare={UNCERTAINTY_THRESH_RARE}  "
          f"thresh_maj={UNCERTAINTY_THRESH_MAJORITY}")
    print("=" * 60)

    retriever = KGRetriever(
        kg, rare_ids=rare_ids,
        top_k=KG_TOP_K, top_nodes=KG_TOP_NODES, hop=KG_HOP,
        lambda_boost=lambda_boost,
    )
    kg_model = KGAugmentedModel(
        base_model=base_model, kg=kg,
        rare_ids=rare_ids, retriever=retriever,
    )
    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )
    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                      split_name="dev",
                                      measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG v2 (GPU)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"],  "dev",  rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"],  "dev",  rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                       split_name="test",
                                       measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT+BiLSTM+MHA+CRF + Rare-Exclusive KG-RAG v2 (GPU)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+Rare-Exclusive-KG-RAG-v2-GPU",
        "kg_config": {
            "lambda_boost":            lambda_boost,
            "gamma_r_boost":           KG_RARE_GAMMA,
            "virtual_alphas":          KG_VIRTUAL_ALPHAS,
            "prototype_k":             KG_PROTOTYPE_K,
            "max_virtual_per_label":   KG_MAX_VIRTUAL_PER_LABEL,
            "uncertainty_thresh_rare": UNCERTAINTY_THRESH_RARE,
            "uncertainty_thresh_maj":  UNCERTAINTY_THRESH_MAJORITY,
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES, "hop": KG_HOP,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF
               + Rare-Exclusive Dense KG + Biased Retrieval (GPU)

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDE

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 37.4s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 37.8s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 37.6s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 37.3s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val